In [2]:
import BarcaUCL_conditional_prob as lib
import numpy as np
# match operator function and monte carlo and visualizer, and im done!

In [3]:
# here is barca's laliga run
# Format: (Opponent, Relative_Day, is_home)
laliga_fixtures = [
    ("Getafe", 5, True),        # March 15
    ("Atletico", 12, False),    # March 22
    ("Celta", 26, True),       # April 5
    ("Osasuna", 33, False),     # April 12
    ("Real Madrid", 40, True),  # April 19 - EL CLÁSICO
    ("Betis", 47, False),       # April 26
    ("Rayo Vallecano", 54, True),# May 3
    ("Espanyol", 61, False),    # May 10
    ("Alaves", 68, True),       # May 17
    ("Sevilla", 75, False)      # May 24 - Final Day
]
big_three_names=["Barcelona", "Real Madrid", "Atletico Madrid"]

In [ ]:
# # Now lets build a function that handles laliga games
# # This function is g
# def condition_joint(prior_prob, t1, t2, w=0.1):
#     f1 = list(t1.form_numerical.values()) # [L, D, W]
#     f2 = list(t2.form_numerical.values()) # [L, D, W]
#
#     # Team 1 wins boost index 0, Team 2 wins boost index 2
#     # Team 1 losses boost index 2, Team 2 losses boost index 0
#     adj = np.array([
#         np.arctan(f1[2] + f2[0]), # Total 'Team 1 Win' momentum
#         np.arctan(f1[1] + f2[1]), # Total 'Draw' momentum
#         np.arctan(f1[0] + f2[2])  # Total 'Team 2 Win' momentum
#     ]) * w
#
#     new_prob = prior_prob + adj
#     return new_prob / new_prob.sum()
#
# def condition_prob_single(prior_prob, team, is_home, w=0.1):
#     f = list(team.form_numerical.values())
#     # Align form [L, D, W] to [Opp1, Draw, Opp2]
#     if is_home: # Team is Opp1
#         adj = np.array([f[2], f[1], f[0]])
#     else: # Team is Opp2
#         adj = np.array([f[0], f[1], f[2]])
#
#     new_prob = prior_prob + (np.arctan(adj) * w)
#     return new_prob / new_prob.sum()


In [26]:
# Now lets build a function that handles laliga games
# This function is g
def condition_joint(prior_prob, t1, t2, w=0.1):
    f1 = list(t1.form_numerical.values()) # [L, D, W]
    f2 = list(t2.form_numerical.values()) # [L, D, W]

    # Team 1 wins boost index 0, Team 2 wins boost index 2
    # Team 1 losses boost index 2, Team 2 losses boost index 0
    adj = np.array([
        np.arctan(f1[2] + f2[0]), # Total 'Team 1 Win' momentum
        np.arctan(f1[1] + f2[1]), # Total 'Draw' momentum
        np.arctan(f1[0] + f2[2])  # Total 'Team 2 Win' momentum
    ]) * w

    new_prob = prior_prob + adj
    return new_prob / new_prob.sum()

def condition_prob_single(prior_prob, team, is_home, w=0.1):
    f = list(team.form_numerical.values())
    f = np.array(f)
    new_prob = prior_prob + (np.arctan(f) * w)
    return new_prob / new_prob.sum()


def match_operator_laliga(big_three, opp , draw_gap = 88):
    # let us compute prior probability of the game using elo and draw gap logic
    delta = abs(big_three.elo - opp.elo)
    if big_three.name == "Barcelona":
        diff_elo = abs(big_three.elo - opp.elo)
        if diff_elo < 100:  # big game
            delta = big_three.assemble(is_big_matchday = True)- opp.assemble()
    elif opp.name == "Barcelona":
        diff_elo = abs(big_three.elo - opp.elo)
        if diff_elo < 100:  # big game
            delta = big_three.assemble()- opp.assemble(is_big_matchday = True)
    else:
        delta = abs(big_three.assemble() - opp.assemble())

    p_big_three = 1 / (1 + 10**(-(delta - draw_gap) / 400))
    p_opp = 1 / (1 + 10**((delta + draw_gap) / 400))
    p_draw = 1 - p_opp - p_big_three
    prior_prob = np.array([p_opp, p_draw, p_big_three]) # macthes the order of self.form

    # 3. Spanish Team Conditioning
    # Check which teams are in the Big Three to apply form
    is_1_big = big_three.name in big_three_names
    is_2_big = opp.name in big_three_names
    if is_1_big and is_2_big:
        # Joint conditioning for two giants
        conditioned_prob = condition_joint(prior_prob, big_three, opp)
    elif is_1_big or is_2_big:
        conditioned_prob = condition_prob_single(prior_prob, big_three, is_home=False)
    else:
        conditioned_prob = prior_prob

    # 1. Determine the Class Factor for this match
    # We can average the class values of both teams
    current_match_class = (big_three.class_ + opp.class_)/2
    # 2. Dirichlet Alpha parameters
    alphas = conditioned_prob* current_match_class
    # First, sample the match-day probability vector
    stochastic_probs = np.random.dirichlet(alphas)
    res_mult = decide_winner_mutlinomial(stochastic_probs)
    res_poisson = decide_winner_poisson(stochastic_probs)
    return res_mult, res_poisson[0]

In [28]:
def decide_winner_mutlinomial(prob):
    # now conditioned_prob will be put to ditrect distribution and it will be sampled

    # Finally, sample the actual result from the multinomial distribution
    # [1, 0, 0] = Home Win, [0, 1, 0] = Draw, [0, 0, 1] = Away Win
    final_result = np.random.multinomial(1, prob)
    if final_result[-1]:
        return "W"
    elif final_result[1]:
        return "D"
    else:
        return "L"

def condition_prob(prior_prob,f, w = 0.1):
    # form is always loss draw win
    form = f.copy()
    arc_form = np.arctan(form) * w
    adjusted_prob = arc_form + prior_prob
    return adjusted_prob/np.sum(adjusted_prob)

In [29]:
def decide_winner_poisson(stochastic_probs):
    """
    stochastic_probs: The [p_opp, p_draw, p_big_three] vector
    already fine-tuned by your Dirichlet model.
    """
    # We use a base scoring rate for a high-level match (e.g., 2.8 goals total)
    base_lambda = 2.8

    # Map probabilities to expected goals
    # Team A (Opponent) lambda is influenced by their win + half the draw prob
    lambda_opp = base_lambda * (stochastic_probs[0] + 0.5 * stochastic_probs[1])
    # Team B (Big Three) lambda
    lambda_big_three = base_lambda * (stochastic_probs[2] + 0.5 * stochastic_probs[1])

    # Sample actual goals from the Poisson distribution
    goals_opp = np.random.poisson(lambda_opp)
    goals_big_three = np.random.poisson(lambda_big_three)

    # Return the result for your update_points and UCL tree logic
    if goals_big_three > goals_opp:
        return "W", goals_big_three, goals_opp
    elif goals_big_three == goals_opp:
        return "D", goals_big_three, goals_opp
    else:
        return "L", goals_big_three, goals_opp


# The "Stress Test" Fixture List
# Each tuple: (HomeTeamObject, AwayTeamObject, is_big_match)

In [32]:
teams_to_test =  list(lib.read_teams().keys())
def sample_teams():

    spanish_bias = 5
    weights = np.array([spanish_bias if name in big_three_names else 1.0 for name in teams_to_test])
    sampling_probs = weights / weights.sum()
    t1_name, t2_name = np.random.choice(teams_to_test, size=2, replace=False, p=sampling_probs)
    fixture_id = tuple(sorted((t1_name, t2_name)))
    return fixture_id

def simulate_random_games():
    comparison_data = []
    fixtures = set()
    for i in range(50):
        outcome_dict = {"D": 0, "W": 1, "L": -1}
        fixture_id = sample_teams()
        while fixture_id in fixtures:
            fixture_id = sample_teams()
        t1_name, t2_name = fixture_id
        b_1 = [t1_name == "Barcelona", t1_name == 'Real Madrid', t1_name == 'Atletico']
        b_2 = [t2_name == "Barcelona", t2_name == 'Real Madrid', t2_name == 'Atletico']
        opp1 = lib.Team(t1_name, is_barca = b_1[0], is_madrid = b_1[1] , is_atletico = b_1[2])
        opp2 = lib.Team(t2_name, is_barca = b_2[0], is_madrid = b_2[1] , is_atletico = b_2[2])
        s_1 = sum(b_1)
        s_2 = sum(b_2)
        res_mult, res_poisson = None, None
        if s_1:
            res_mult, res_poisson = match_operator_laliga(big_three = opp1, opp = opp2)
        if s_2:
            res_mult, res_poisson = match_operator_laliga(big_three = opp2, opp = opp1)
        else:
            res_mult, res_poisson = match_operator_laliga(opp2, opp1)
        res_mult_num = outcome_dict[res_mult]
        res_poisson_num = outcome_dict[res_poisson]
        comparison_data.append([res_mult_num, res_poisson_num])
    return np.array(comparison_data)


In [33]:

def run_covariance_test(iterations=1000):
    all_correlations = []
    for i in range(iterations):
        comp = simulate_random_games()
        # We check if there is variance in the results to avoid NaN in correlation
        if np.var(comp[:, 0]) == 0 or np.var(comp[:, 1]) == 0:
            continue
        try:
            correlation = np.corrcoef(comp.T)[0, 1]
            if not np.isnan(correlation):
                all_correlations.append(correlation)
        except:
            continue
        correlation = np.corrcoef(comp.T)[0, 1]
        all_correlations.append(correlation)
    mean_corr = np.mean(all_correlations)
    std_corr = np.std(all_correlations)

    print(f"--- Analysis Complete ---")
    print(f"Mean Correlation: {mean_corr:.4f}")
    print(f"Standard Deviation: {std_corr:.4f}")

run_covariance_test(iterations=1000)

AttributeError: 'Team' object has no attribute 'form_numerical'

In [ ]:


def match_operator_ucl(opp1, opp2, ):
    """
    Simulates a UCL knockout match.
    Handles joint conditioning if both teams are in the 'Big Three'.
    """
    # 1. Delta Calculation (Skill Gap)
    # If it's Barca, we trigger their 'big_matchday' assembly
    if opp1.name == "Barcelona":
        delta = opp1.assemble(is_big_matchday=True) - opp2.assemble()
    elif opp2.name == "Barcelona":
        delta = opp1.assemble() - opp2.assemble(is_big_matchday=True)
    else:
        # We don't use abs() here because the direction of delta
        # defines who is the favorite in the logistic function
        delta = opp1.assemble() - opp2.assemble()

    # 2. Prior Probabilities (Logistic)
    # We remove the draw gap for UCL to keep the '90 min' draw realistic
    p_opp1 = 1 / (1 + 10**(-delta / 400))
    p_opp2 = 1 / (1 + 10**(delta / 400))
    # In the UCL, the chance of a 90-min draw is naturally higher due to tension
    # We'll set a base 25% draw chance and normalize
    p_draw = 0.25   # needs to be handled
    prior_prob = np.array([p_opp1, p_draw, p_opp2])
    prior_prob /= prior_prob.sum()

    # 3. Spanish Team Conditioning
    # Check which teams are in the Big Three to apply form
    is_1_big = opp1.name in big_three_names
    is_2_big = opp2.name in big_three_names

    if is_1_big and is_2_big:
        # Joint conditioning for two giants
        conditioned_prob = condition_joint(prior_prob, opp1, opp2)
    elif is_1_big:
        conditioned_prob = condition_prob_single(prior_prob, opp1, is_home=True)
    elif is_2_big:
        conditioned_prob = condition_prob_single(prior_prob, opp2, is_home=False)
    else:
        conditioned_prob = prior_prob

    # 4. Determine Match Result
    current_match_class = (opp1.class_ + opp2.class_) / 2
    res = decide_winner(conditioned_prob, current_match_class)

    # 5. Handle Draw (Penalty Shootout)
    if res == "D":
        # Penalties weighted by Class Factor (Higher class = more likely to win pens)
        # Probability of opp1 winning pens
        prob_opp1_pens = opp1.class_ / (opp1.class_ + opp2.class_)
        return opp1 if np.random.uniform(0, 1) < prob_opp1_pens else opp2

    return opp1 if res == "W1" else opp2


def decide_winner(prob, class_value):
    alphas = prob * class_value
    stochastic_probs = np.random.dirichlet(alphas)
    final_result = np.random.multinomial(1, stochastic_probs)

    if final_result[0]: # Opp1 wins
        return "W1"
    elif final_result[1]: # Draw
        return "D"
    else: # Opp2 wins
        return "W2"